<a href="https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/falah-bit/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Question types to pick from: label provenance (is the outcome observed, or defined
by a rule that makes the finding circular?), validation design (grouped/time-aware
split, or could train/test overlap inflate the result?), sample size / support,
generalization scope, confounds.

### Finding 1

- **Paper claim:** "What Predicts Growth?" (ML Appendix — Growth & Classification, p.28) — a Logistic Regression model, evaluated on an 80/20 holdout split, achieves 71% accuracy separating growing pages from declining pages, with Content Age as the strongest negative signal and Days Visible / recent Impressions as the strongest positive signals.
- **My methodology question:** The Methodology section states the split as "80/20" but doesn't say whether it's grouped by client/site or purely random row-level. If pages from the same client can land in both train and test, client-specific patterns (template, niche, existing SEO maturity) could leak across the split and inflate the 71% accuracy above what it would be on a genuinely unseen client.
- **Why I'm asking:** This is the exact check I ran on my own Week-5 model (Section 2 of this notebook) — a naive random split versus a client-grouped split gave meaningfully different, and more trustworthy, numbers. If the paper's split was already grouped, saying so explicitly would make the 71% easier to trust at face value; if not, re-running with a client-grouped holdout (the same test I applied to myself) would show whether the number holds up.

### Finding 2

- **Paper claim:** "What Predicts Health?" (ML Appendix — Feature Importance, p.27) — a Random Forest finds Average Position (43%) and Impressions (32%) as the top predictors of Health Score. The paper itself notes: "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal."
- **My methodology question:** Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — so Position and Impressions are literal components of the label, not independent predictors of it. Given that, how much of the 43%/32% importance reflects genuine external signal versus the mechanical fact that these features are baked into the label's formula?
- **Why I'm asking:** The paper already flags this caveat, which is good practice — my question just pushes one step further, toward the same sensitivity-check habit I used on my own model (Section 3: refitting without `avg_position` to see what's left). A version of this Random Forest run with Position/Impressions held out would show what the model actually learns once the built-in overlap is removed.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

Moving from a naive random split to a grouped split changes the picture substantially: 43 of 46 clients appeared in both train and test under the naive split, meaning client-specific patterns were shared across the split almost entirely. Under the naive split, F1 was actually lower (0.510) than under the grouped split (0.642), and precision was much lower (0.389 vs 0.586) despite the leakage — so the leakage didn't simply inflate every metric. Still, with 43/46 clients shared, the naive numbers cannot be read as an estimate of performance on genuinely unseen clients. The grouped split, with zero client overlap, is the one I trust.




In [6]:
%pip install -q duckdb huggingface_hub pandas numpy scikit-learn

import duckdb, os
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    )
""")

MONTH_FEATURES = "2026-03"
MONTH_OUTCOME  = "2026-04"
TABLE_URI_FEAT = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH_FEATURES}/*.parquet"
TABLE_URI_OUT  = f"hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH_OUTCOME}/*.parquet"

# --- rebuild the same feature table and label as w05_model.ipynb ---
df = con.execute(f"""
    SELECT
        content_hash_id, client_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks)       AS total_clicks,
        SUM(gsc_impressions)  AS total_impressions,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr
    FROM read_parquet('{TABLE_URI_FEAT}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()
df["avg_ctr"] = df["avg_ctr"].fillna(0)

df_outcome = con.execute(f"""
    SELECT content_hash_id, client_hash_id, AVG(gsc_avg_position) AS avg_position_next
    FROM read_parquet('{TABLE_URI_OUT}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

labeled = df.merge(df_outcome, on=["content_hash_id", "client_hash_id"], how="inner")
POSITION_WORSEN_THRESHOLD = 3  # same threshold as w05, justified there
position_delta = labeled["avg_position_next"] - labeled["avg_position"]
labeled["is_declining_label"] = (position_delta >= POSITION_WORSEN_THRESHOLD).astype(int)

FEATURES = ["avg_position", "total_clicks", "total_impressions", "avg_ctr"]
print("Rows:", len(labeled), "| positive rate:", labeled["is_declining_label"].mean().round(3))

def fit_and_score(train, test, name):
    X_train, y_train = train[FEATURES].fillna(0), train["is_declining_label"]
    X_test, y_test   = test[FEATURES].fillna(0), test["is_declining_label"]
    model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
    pred, proba = model.predict(X_test), model.predict_proba(X_test)[:, 1]
    overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
    return {
        "split": name,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
        "clients_in_both_train_and_test": len(overlap),
    }, model

# --- BEFORE: naive random split (ignores client grouping) ---
train_naive, test_naive = train_test_split(
    labeled, test_size=0.2, random_state=42, stratify=labeled["is_declining_label"]
)
before, _ = fit_and_score(train_naive, test_naive, "Before (naive random split)")

# --- AFTER: grouped split by client_hash_id (honest) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(labeled, groups=labeled["client_hash_id"]))
train_grouped, test_grouped = labeled.iloc[train_idx].copy(), labeled.iloc[test_idx].copy()
after, model_after = fit_and_score(train_grouped, test_grouped, "After (grouped by client)")

comparison = pd.DataFrame([before, after])
print(comparison)

# --- real failure examples from the honest (after) split ---
X_test_after = test_grouped[FEATURES].fillna(0)
y_test_after = test_grouped["is_declining_label"]
pred_after = model_after.predict(X_test_after)
fp = test_grouped[(y_test_after.values == 0) & (pred_after == 1)]
fn = test_grouped[(y_test_after.values == 1) & (pred_after == 0)]
print(f"\nFalse positives: {len(fp)} | False negatives: {len(fn)}")
print(fn[FEATURES].sample(min(5, len(fn)), random_state=1))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549 | positive rate: 0.345
                         split  precision    recall        f1   roc_auc  \
0  Before (naive random split)   0.391547  0.738459  0.511752  0.585384   
1    After (grouped by client)   0.585717  0.709287  0.641607  0.556929   

   clients_in_both_train_and_test  
0                              44  
1                               0  

False positives: 5807 | False negatives: 3365
        avg_position  total_clicks  total_impressions   avg_ctr
103864     32.166667           0.0               22.0  0.000000
147539     32.147593           1.0               82.0  0.012195
82479      13.622063           6.0             1233.0  0.004866
17956      30.682667           2.0             5049.0  0.000396
108286     34.408976           0.0              457.0  0.000000


## 3. Leakage audit

No outcome-month column made it into FEATURES, so there's no temporal leakage feeding the model. The real leakage risk was client overlap: the naive split shared 43 of 46 clients between train and test, while the grouped split had zero overlap by design — confirming the naive split's numbers can't be used as an unseen-client estimate. The avg_position sensitivity check is reassuring: removing it barely moves the results (F1 0.642 → 0.659, ROC-AUC 0.557 → 0.537), so the model's apparent skill isn't a mechanical artifact of avg_position appearing in both the label formula and the features. The one thing I'd flag as a caveat, not a bug: ML-07's expected_ctr-per-tier is computed on the full dataset, which is fine since it isn't used as a feature here — but it's worth remembering if a future model does use it.

In [7]:
# --- Check 1: temporal leakage ---
# All FEATURES come from MONTH_FEATURES; the label uses avg_position_next from
# MONTH_OUTCOME, which is never joined into the feature columns.
outcome_only_cols = {"avg_position_next"}
print("Any outcome-month column in FEATURES?", bool(set(FEATURES) & outcome_only_cols))

# --- Check 2: client leakage across splits ---
overlap_grouped = set(train_grouped["client_hash_id"]) & set(test_grouped["client_hash_id"])
overlap_naive = set(train_naive["client_hash_id"]) & set(test_naive["client_hash_id"])
print(f"Client overlap - grouped split: {len(overlap_grouped)}")
print(f"Client overlap - naive split:   {len(overlap_naive)} "
      f"(out of {labeled['client_hash_id'].nunique()} total)")

# --- Check 3: avg_position is used both as a feature and inside the label formula
# (label depends on avg_position_next - avg_position). Not temporal leakage (known
# at prediction time), but can create a mechanical relationship. Sensitivity check:
# refit without avg_position.
from sklearn.linear_model import LogisticRegression as LR
X_train_sens = train_grouped[["total_clicks", "total_impressions", "avg_ctr"]].fillna(0)
X_test_sens  = test_grouped[["total_clicks", "total_impressions", "avg_ctr"]].fillna(0)
y_train_sens = train_grouped["is_declining_label"]
y_test_sens  = test_grouped["is_declining_label"]
model_sens = LR(max_iter=1000, class_weight="balanced").fit(X_train_sens, y_train_sens)
pred_sens  = model_sens.predict(X_test_sens)
proba_sens = model_sens.predict_proba(X_test_sens)[:, 1]

print(f"\nF1 with avg_position:    {after['f1']:.3f}")
print(f"F1 without avg_position: {f1_score(y_test_sens, pred_sens, zero_division=0):.3f}")
print(f"ROC-AUC with avg_position:    {after['roc_auc']:.3f}")
print(f"ROC-AUC without avg_position: {roc_auc_score(y_test_sens, proba_sens):.3f}")

# --- Check 4: note on full-dataset vs train-only stats ---
print("\n[Note] ML-07/w05's expected_ctr-per-position-tier is computed on the full "
      "dataset across all clients - fine for that hand-written rule (not a trained "
      "model), and it is NOT included in FEATURES here, so it does not leak into "
      "this model's train/test evaluation.")


Any outcome-month column in FEATURES? False
Client overlap - grouped split: 0
Client overlap - naive split:   44 (out of 46 total)

F1 with avg_position:    0.642
F1 without avg_position: 0.659
ROC-AUC with avg_position:    0.557
ROC-AUC without avg_position: 0.537

[Note] ML-07/w05's expected_ctr-per-position-tier is computed on the full dataset across all clients - fine for that hand-written rule (not a trained model), and it is NOT included in FEATURES here, so it does not leak into this model's train/test evaluation.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (w05):** "Logistic Regression beats the baseline clearly on F1 (0.642 vs
0.502) ... a genuinely useful improvement."

**Rewritten (safe):** In the grouped test split used here (10 clients, one
month-over-month window, 2026-03 to 2026-04), Logistic Regression showed a higher
*measured* F1 than the baseline rule. This is a *directional, decision-support*
signal from a single time window and a small client sample - not a claim that the
model generalizes to other months, other clients, or a larger holdout.

**Original (w05):** "Random Forest ... underperforms the baseline on F1 ... a clear
case where added model complexity did not translate into better decisions."

**Rewritten (safe):** In this same observed split, Random Forest's measured F1 was
lower than both the baseline and Logistic Regression. This is directional evidence
against the more complex model for this particular task and split - not a general
claim about Random Forest versus Logistic Regression beyond this data.

**Original (w05):** "`total_clicks` is by far the strongest driver of the model's predictions... `avg_ctr` — the signal the baseline rule leans on almost entirely — has essentially no predictive power for future decline."

**Rewritten (safe):** In this observed test split, `total_clicks` showed the largest measured permutation importance for the Logistic Regression model, while `avg_ctr` showed close to none. This is a directional finding from one model on one split — not a claim that CTR is universally uninformative for predicting decline.

In [8]:
# Numbers referenced by the claim rewrite above, printed for traceability
print("Baseline vs model F1 comparison this claim rewrite is based on:")
print(comparison[["split", "f1", "roc_auc", "clients_in_both_train_and_test"]])


Baseline vs model F1 comparison this claim rewrite is based on:
                         split        f1   roc_auc  \
0  Before (naive random split)  0.511752  0.585384   
1    After (grouped by client)  0.641607  0.556929   

   clients_in_both_train_and_test  
0                              44  
1                               0  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.